In [ ]:
import sys
sys.path.insert(0, '/Users/josephledford/pycaret-env/lib/python3.11/site-packages')

In [ ]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
# makes sure that when I ouput the dataset it shows all the columns
pd.set_option('display.max_rows', 90)
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning, module='pandas.io.formats.format')

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('darkgrid')


from sklearn.neighbors import KNeighborsRegressor
import scipy.stats
from sklearn.preprocessing import StandardScaler
from pycaret.regression import setup, compare_models
from sklearn.model_selection import KFold, cross_val_score

import catboost
from catboost import CatBoostRegressor
from sklearn.linear_model import BayesianRidge, HuberRegressor, Ridge, OrthogonalMatchingPursuit
from lightgbm import LGBMRegressor
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)


In [ ]:
train0 = pd.read_csv('/Users/josephledford/Downloads/Housing Prices Competition/train.csv')
test0 = pd.read_csv('/Users/josephledford/Downloads/Housing Prices Competition/test.csv')
sample_submission = pd.read_csv('/Users/josephledford/Downloads/Housing Prices Competition/sample_submission.csv')

In [ ]:
train0

In [ ]:
test0

In [ ]:
sample_submission

# Combine Train and Test Sets

In [ ]:
# After preprocessing, split the sets back
# Making sure to not shuffle the data so the training sets don't get mixed up

# Need to strip training set SalePrice becasue it is all NaN 
target = train0['SalePrice']
test_ids = test0['Id']

# Train0 is raw data, train1 is one step up from the raw data
train1 = train0.drop(['Id', 'SalePrice'], axis=1)
test1 = test0.drop('Id', axis=1)

data1 = pd.concat([train1, test1], axis=0).reset_index(drop=True) # axis = 0 means stacking on top by index(rows)down each column
data1
# Now you can look at more data for preprocessing because you added the test data.

In [ ]:
target

# Cleaning

In [ ]:
data2 = data1.copy()

## Ensure Proper Data Types

In [ ]:
data2['MSSubClass'] = data2['MSSubClass'].astype(str)

## Fill Categorical Missing Values

In [ ]:
# Could take awhile
# Must go through each varaible and determine when to fill with the mode or whether a particular feature has valid missing values.
# The variable Ally uses NA for "No alley acess" which is a valid value
# The variable BsmtFinType1 has NA for "No Basement"

# If a variable has an NA value: impute using constant
# If a variable doesn't have an NA value: impute using column mode

# Impute using a constant value
for column in [
    'Alley',
    'BsmtQual',
    'BsmtCond',
    'BsmtExposure',
    'BsmtFinType1',
    'BsmtFinType2',
    'FireplaceQu',
    'GarageType',
    'GarageFinish',
    'GarageQual',
    'GarageCond',
    'PoolQC',
    'Fence',
    'MiscFeature'   
]:
    data2[column] = data2[column].fillna("None")

# Impute using the column mode
for column in [
    'MSZoning',
    'Utilities',
    'Exterior1st',
    'Exterior2nd',
    'MasVnrType',
    'Electrical',
    'KitchenQual',
    'Functional',
    'SaleType'  
]:
    data2[column] = data2[column].fillna(data2[column].mode()[0])

In [ ]:
data3 = data2.copy()

## Numeric Missing Values

In [ ]:
# Use K-Nearest Neighbors (KNN) imputation - looks at relationships between different features to figure out what value it should be
# Function takes in a dataframe and a column and will return the same dataframe with the columns missing values filled in.
# Uses the variables that don't have missing values as the neighbors for the target column you are trying to fill
def knn_impute(df, na_target):
    df = df.copy()
    
    numeric_df = df.select_dtypes(np.number) # numerical colmns only 
    non_na_columns = numeric_df.loc[:, numeric_df.isna().sum() == 0].columns # columns with no missing values

    y_train = numeric_df.loc[numeric_df[na_target].isna() == False, na_target] # all of the values of na_target that are not missing values
    X_train = numeric_df.loc[numeric_df[na_target].isna() == False, non_na_columns]  # all of the values of the rest of the data that do not have missing values for the na_target
    X_test = numeric_df.loc[numeric_df[na_target].isna() == True, non_na_columns] # all of the values of the rest of the data that do have missing values for the na_target
    
    knn = KNeighborsRegressor()
    knn.fit(X_train, y_train)

    y_pred = knn.predict(X_test) # values used as imputation

    df.loc[df[na_target].isna() == True, na_target] = y_pred
    
    return df

In [ ]:
for column in [
    'LotFrontage',
    'MasVnrArea',
    'BsmtFinSF1',
    'BsmtFinSF2',
    'BsmtUnfSF',
    'TotalBsmtSF',
    'BsmtFullBath',
    'BsmtHalfBath',
    'GarageYrBlt',
    'GarageCars',
    'GarageArea'
]:
    data3 = knn_impute(data3, column)

In [ ]:
data4 = data3.copy()

# Feature Transformations

## Log Transform for Skewed Features

In [ ]:
# Dataframe of numeric columns labeled Feature
skew_df = pd.DataFrame(data4.select_dtypes(np.number).columns, columns=['Feature'])
# New column of Skewness for each feature 
skew_df['Skew'] = skew_df['Feature'].apply(lambda feature: scipy.stats.skew(data4[feature]))
skew_df['Absolute Skew'] = skew_df['Skew'].apply(abs)
skew_df['Skewed'] = skew_df['Absolute Skew'].apply(lambda x: True if x >= 0.5 else False)
skew_df

In [ ]:
for column in skew_df.query("Skewed == True")['Feature'].values:
    data4[column] = np.log1p(data4[column])

In [ ]:
# After log transformation
skew_df = pd.DataFrame(data4.select_dtypes(np.number).columns, columns=['Feature'])
skew_df['Skew'] = skew_df['Feature'].apply(lambda feature: scipy.stats.skew(data4[feature]))
skew_df['Absolute Skew'] = skew_df['Skew'].apply(abs)
skew_df['Skewed'] = skew_df['Absolute Skew'].apply(lambda x: True if x >= 0.5 else False)
skew_df

## Cosine Transform for Cyclical Features

In [ ]:
# Sinusoidal transformation weighted by 0.5326 so that 
# 1 & 12 are linked as cold seasons and 6
# is linked to the hottest season
data4['MoSold'] = (-np.cos(0.5326 * data4['MoSold']))
print(np.min(-np.cos(0.5326 * data4['MoSold'])))
print(np.max(-np.cos(0.5326 * data4['MoSold'])))
# min should be super close to negative one (see notes to confirm)
# max should be super close to one

In [ ]:
data5 = data4.copy()

# Encode Categoricals

In [ ]:
data5 = pd.get_dummies(data5)

In [ ]:
data6 = data5.copy()

# Scaling

In [ ]:
scaler = StandardScaler()
scaler.fit(data6)

data6 = pd.DataFrame(scaler.transform(data6), index = data6.index, columns= data6.columns)

In [ ]:
data6

In [ ]:
data7 = data6.copy()

# Target Transformation

In [ ]:
plt.figure(figsize=(20,10))

plt.subplot(1, 2, 1)
sns.distplot(target, kde=True, fit=scipy.stats.norm)
plt.xlabel("SalePrice")
plt.title("Without Log Transform")
# Kde is kernel density estimation which is what gives the graph a line
# fit lets you add a paramter value tries to fit it to the plot
# this fit specification has not been added to the new histplot which is why the deprecated version is being used
plt.subplot(1,2,2)
sns.distplot(np.log(target), kde=True, fit=scipy.stats.norm) # confirm the min of target is not zero, otherwise you need to use log1p
plt.xlabel("Log SalePrice")
plt.title("With Log Transform")
# Fit is now much closer to the data

plt.show()

In [ ]:
log_target = np.log(target)

## Split Data

In [ ]:
train_final = data7.loc[:train0.index.max(),:].copy()
test_final = data7.loc[train0.index.max() + 1:, :].reset_index(drop=True).copy()

In [ ]:
train_final

In [ ]:
test_final

In [ ]:
_ = setup(data=pd.concat([train_final, log_target], axis=1), target='SalePrice')

In [ ]:
compare_models()

# Hyperparameter Optimization

In [ ]:
 kf = KFold(n_splits=10)

In [ ]:
# hyperparameter optimization for the bayesian ridge model
# parameters found on scikit-learn documentation
# look at each parameter and its default value, cross referece with the methods from the
# optuna documentation, then impletment the correct optuna method working around the default
# values given for each parameter to create a plausible range of suggestions
def br_objective(trail):
    n_iter = trail.suggest_int('n_iter', 50, 600)
    tol = trail.suggest_loguniform('tol', 1e-8, 10)
    alpha_1 = trail.suggest_loguniform('alpha_1', 1e-8, 10.0)
    alpha_2 = trail.suggest_loguniform('alpha_2', 1e-8, 10.0)
    lambda_1 = trail.suggest_loguniform('lambda_1', 1e-8, 10.0)
    lambda_2 = trail.suggest_loguniform('lambda_2', 1e-8, 10.0)

# Now apply the optuna hyperparemeters to a standard BayesianRidge model
    model = BayesianRidge(
        n_iter = n_iter,
        tol = tol,
        alpha_1 = alpha_1,
        alpha_2 = alpha_2,
        lambda_1 = lambda_1,
        lambda_2 = lambda_2
    )

    model.fit(train_final, log_target)
   
    cv_scores = np.exp(np.sqrt(-cross_val_score(model, train_final, log_target, scoring='neg_mean_squared_error', cv=kf)))

    return np.mean(cv_scores) # want the mean because otherwise it will return the value from every iteration


In [ ]:
study = optuna.create_study(direction='minimize')
study.optimize(br_objective, n_trials=100)

In [ ]:
study.best_params

In [ ]:
def ridge_objective(trial):
    alpha = trial.suggest_loguniform('alpha', 1, 1000)

    ridge_model = Ridge(
        alpha = alpha
    )

    ridge_model.fit(train_final, log_target)

    ridge_cv_scores = np.exp(np.sqrt(-cross_val_score(ridge_model, train_final, log_target, scoring='neg_mean_squared_error', cv=kf)))

    return np.mean(ridge_cv_scores)


In [ ]:
ridge_study = optuna.create_study(direction='minimize')
ridge_study.optimize(ridge_objective, n_trials = 100)

In [ ]:
ridge_study.best_params

In [ ]:
def cb_objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 0.003, 0.007, log=True)
    depth = trial.suggest_int("depth", 4, 6)
    l2_leaf_reg = trial.suggest_float("l2_leaf_reg", 0.1, 2, log=True)

    cb_model = CatBoostRegressor(
        iterations=6000,
        learning_rate=learning_rate,
        depth=depth,
        l2_leaf_reg=l2_leaf_reg,
        eval_metric="RMSE",
        early_stopping_rounds=200,
        random_seed=42,
        verbose=False
    )

    cb_model.fit(train_final, log_target)
    
    cb_cv_scores = cross_val_score(
        cb_model, 
        train_final, 
        log_target, 
        scoring='neg_mean_squared_error', 
        cv=kf
    )
    
    return np.exp(np.sqrt(-cb_cv_scores.mean()))


In [ ]:
cb_study = optuna.create_study(direction='minimize')
cb_study.optimize(cb_objective, n_trials = 100)

In [ ]:
cb_study.best_params

In [ ]:
def lgbm_objective(trial):
    num_leaves = trial.suggest_int('num_leaves', 10, 100)
    max_depth = trial.suggest_int('max_depth', 0, 10)
    learning_rate = trial.suggest_loguniform('learning_rate', 0.001, 0.3)
    n_estimators = trial.suggest_int('n_estimators', 50, 600)

    lgbm_model = LGBMRegressor(
        num_leaves = num_leaves,
        max_depth = max_depth,
        learning_rate = learning_rate,
        n_estimators = n_estimators
    )

    lgbm_model.fit(train_final, log_target)

    lgbm_cv_scores = np.exp(np.sqrt(-cross_val_score(lgbm_model, train_final, log_target, scoring='neg_mean_squared_error', cv=kf)))

    return np.mean(lgbm_cv_scores)




In [ ]:
lgbm_study = optuna.create_study(direction='minimize')
lgbm_study.optimize(lgbm_objective, n_trials = 100)

In [ ]:
lgbm_study.best_params

# Bagging Ensemble

In [ ]:
br_params = {
    'n_iter': 225,
    'tol': 0.5156853223609507,
    'alpha_1': 2.3288877481113583e-05,
    'alpha_2': 9.24631090979403,
    'lambda_1': 9.484902038544837,
    'lambda_2': 2.2346955260531906e-08
}

ridge_params = {
    'alpha': 346.2355401495992
}

cb_params = {
    'learning_rate': 0.006267794504232028,
    'depth': 5,
    'l2_leaf_reg': 0.420258564938903,
    'eval_metric': 'RMSE',
    'early_stopping_rounds': 200,
    'random_seed': 42
}

lightgbm_params = {
    'num_leaves': 63,
    'max_depth': 3,
    'learning_rate': 0.06039580203944131,
    'n_estimators': 580
}

In [ ]:
# The ** unpacks the dictionary of optimized parameters and imposes it on the respective model
models = {
    "catboost" : CatBoostRegressor(**cb_params, verbose=0),
    "br": BayesianRidge(**br_params),
    "lightgbm": LGBMRegressor(**lightgbm_params),
    "ridge": Ridge(**ridge_params),
    "omp": OrthogonalMatchingPursuit() # not enough to warrant hyperparameter optimization
}

In [ ]:
for name, model in models.items():
    model.fit(train_final, log_target)
    print(name + "trained.")

# Evaluate

In [ ]:
results = {}


for name, model in models.items():
    result = np.exp(np.sqrt(-cross_val_score(model, train_final, log_target, scoring='neg_mean_squared_error', cv=kf)))
    results[name] = result # passes result to dictionary using the name

# do - of the cross-validation score so you can see the proper mse
# you square root it so that you get the ROOT Mean Sqaured Error that the competition designates
# np.exp() makes sure your result acurrately represents the target given, not the log_target we made

In [ ]:
results 

In [ ]:
# Summary Statistics
for name, result in results.items():
    print("-----------\n" + name + "\n------------") # Separates results for each model
    print(np.mean(result))
    print(np.std(result))

# Combine Predictions

In [ ]:
final_predictions = (
    0.4 * np.exp(models['catboost'].predict(test_final)) +
    0.2 * np.exp(models['br'].predict(test_final)) +
    0.2 * np.exp(models['lightgbm'].predict(test_final)) +
    0.1 * np.exp(models['ridge'].predict(test_final)) +
    0.1 * np.exp(models['omp'].predict(test_final)) 
)

In [ ]:
final_predictions

# Make Submission

In [ ]:
submission = pd.concat([test_ids, pd.Series(final_predictions, name='SalePrice')], axis = 1)
submission

In [ ]:
submission.to_csv('/Users/josephledford/Downloads/Housing Prices Competition/004_submission.csv', index=False, header=True)